# Detector de Spam

En este notebook vamos a ver como clasificar una serie de correos electrónicos recibidos en nuestra bandeja de entrada.


* Los correos están clasificados como: *spam* o *ham* (no spam)

* El este notebook realizaremos los siguientes pasos:
    
    1. Carga de los datos y transformación (csv)
    2. Normalización
    3. Creación de la Bolsa de Palabras
    4. Particionado de Datos
    5. Creación del modelo Multinomial Naive Bayes
    6. Evaluación de los modelos

## 1.- Carga de datos
Importamos los datos originales y los transformamos para facilitar su utilización.

El fichero original tiene 2 columnas:

*   mensaje y tipo

Tenemos dos ficheros, uno con pocos registros (mensajes_spam_basico.csv) y otro con más registros (mensajes_spam_v2.csv)





In [ ]:
!python -m spacy download es

In [7]:
import pandas as pd
import numpy as np

# 1. Cargar datos del archivo con los mensajes básicos
# df = pd.read_csv('mensajes_spam.csv')

# Cargar datos del archivo con los mensajes ampliado
df = pd.read_csv('mensajes_spam_v2.csv')

# Mostramos las primeras 5 observaciones
df.head()


,mensaje,tipo
0,¡Has sido seleccionado para ganar un iPhone 15...,spam
1,Reclama tu premio ahora antes de que expire la...,spam
2,Gana dinero desde casa trabajando solo 2 horas...,spam
3,Haz clic aquí para duplicar tus ganancias fáci...,spam
4,Oferta exclusiva: consigue un crédito sin inte...,spam


## 2.- Normalización

In [8]:
import spacy

# Cargamos el modelo (asegúrate de haber ejecutado la celda de descarga previa: !python -m spacy download es_core_news_sm)
# Nota: Si da error por el  'es', usamos mnejor el 'es_core_news_sm', ya que es el nombre completo
try:
    nlp = spacy.load('es_core_news_sm')
except:
    nlp = spacy.load('es') # Fallback por si se descargó con el alias corto

def Normalizacion(docs_list):
    corpus_limpio = []
    
    for doc in docs_list:
        # 1. Procesar el documento con spaCy
        tokens = nlp(doc)
        
        # 2. Lematizar y filtrar:
        # - Convertimos a minúsculas (.lower())
        # - Usamos el lema (.lemma_)
        # - Filtramos si es stopword o puntuación
        palabras_limpias = [
            t.lemma_.lower() 
            for t in tokens 
            if not t.is_stop and not t.is_punct and not t.is_space
        ]
        
        # 3. Unir las palabras de nuevo en una cadena de texto
        corpus_limpio.append(" ".join(palabras_limpias))

    return corpus_limpio

# Ejecutamos la normalización sobre la columna de mensajes
# Asegúrate de que 'df' está cargado como se indica en el paso 1 del notebook
corpus = Normalizacion(df['mensaje']) # 'mensaje' es la columna indicada en el PDF [cite: 17]
print("Normalización completada. Ejemplo:", corpus[0])

Normalización completada. Ejemplo: has seleccionar ganar iphone 15 gratis


## 3.- Clasificación y entrenamiento
Despúes del preprocesamiento utilizaremos TF-IDF para la vectorización y Multinomial Naive Bayes para la clasificación.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

# 1. Vectorización
tfidf = TfidfVectorizer()

# Transformamos el corpus limpio en una matriz numérica
X = tfidf.fit_transform(corpus)

# Definimos la variable objetivo (target)
y = df['tipo']

# 2. Particionado de Datos (Train/Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 3. Creación y entrenamiento del modelo Multinomial Naive Bayes
modelo_nb = MultinomialNB()
modelo_nb.fit(X_train, y_train)

# 4. Evaluación de los modelos
y_pred = modelo_nb.predict(X_test)

print("Precisión del modelo:", accuracy_score(y_test, y_pred))
print("\nReporte de Clasificación:\n")
print(classification_report(y_test, y_pred))

Precisión del modelo: 0.7333333333333333

Reporte de Clasificación:

              precision    recall  f1-score   support

         ham       0.75      0.75      0.75         8
        spam       0.71      0.71      0.71         7

    accuracy                           0.73        15
   macro avg       0.73      0.73      0.73        15
weighted avg       0.73      0.73      0.73        15



## 4.- Prueba del modelo entrenado

In [ ]:
# 7. Prueba
# test_review = ["¡Felicidades! Has ganado un premio exclusivo, reclama tu recompensa."]
# test_review = ["Reunión confirmada para las 10 de la mañana."]
# test_review = ["Verifica tu cuenta PayPal inmediatamente para evitar suspensión."]
test_review = ["Vamos a cenar esta noche al club, que hay fiesta."]

# 1. Normalizar el nuevo mensaje
test_limpio = Normalizacion(test_review)

# 2. Vectorizar
test_vector = tfidf.transform(test_limpio)

# 3. Predecir
prediccion = modelo_nb.predict(test_vector)

print(f"Mensaje original: {test_review[0]}")
print(f"Mensaje procesado: {test_limpio[0]}")
print(f"Predicción del modelo: {prediccion[0]}")


Mensaje original: Vamos a cenar esta noche al club, que hay fiesta.
Mensaje procesado: cenar noche club fiesta
Predicción del modelo: ham
